# 03 — Java Platform Threads and Virtual Threads

**Target:** Apache Spark 4.2.0 runtime; a full JDK 21+ with `javac` and `java` on `PATH`; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
How do Java platform threads and virtual threads behave on a blocking workload?

**Why this matters:** Loom can reduce the cost of representing many blocked Java tasks. It does not remove CPU limits or make retained objects free. This lab runs a separate Java program and does not change Spark internals.

In [ ]:
import subprocess, tempfile, pathlib, shutil, sys
print('java:', shutil.which('java'), 'javac:', shutil.which('javac'))
source = r'''
import java.util.concurrent.*;
import java.util.concurrent.atomic.AtomicInteger;
import java.lang.management.*;
public class LoomCompare {
  static void run(String label, ExecutorService pool, int n) throws Exception {
    ThreadMXBean threads=ManagementFactory.getThreadMXBean();
    java.lang.management.OperatingSystemMXBean osBean=ManagementFactory.getOperatingSystemMXBean();
    long cpuBefore=osBean instanceof com.sun.management.OperatingSystemMXBean ? ((com.sun.management.OperatingSystemMXBean)osBean).getProcessCpuTime() : -1;
    long heapBefore=ManagementFactory.getMemoryMXBean().getHeapMemoryUsage().getUsed();
    AtomicInteger peakPlatformThreads=new AtomicInteger(threads.getThreadCount());
    long start=System.nanoTime();
    for (int i=0;i<n;i++) pool.submit(() -> {
      peakPlatformThreads.accumulateAndGet(threads.getThreadCount(), Math::max);
      try { Thread.sleep(100); } catch (InterruptedException e) { Thread.currentThread().interrupt(); }
    });
    pool.shutdown(); pool.awaitTermination(2, TimeUnit.MINUTES);
    long elapsed=TimeUnit.NANOSECONDS.toMillis(System.nanoTime()-start);
    long heapAfter=ManagementFactory.getMemoryMXBean().getHeapMemoryUsage().getUsed();
    long cpuAfter=osBean instanceof com.sun.management.OperatingSystemMXBean ? ((com.sun.management.OperatingSystemMXBean)osBean).getProcessCpuTime() : -1;
    long cpuDelta=cpuBefore >= 0 && cpuAfter >= 0 ? cpuAfter-cpuBefore : -1;
    System.out.printf("%s elapsed_ms=%d approximate_peak_java_platform_threads=%d heap_before=%d heap_after=%d process_cpu_ms=%d%n",
      label,elapsed,peakPlatformThreads.get(),heapBefore,heapAfter,cpuDelta < 0 ? -1 : cpuDelta/1_000_000);
  }
  public static void main(String[] a) throws Exception {
    int n=500;
    run("platform-100",Executors.newFixedThreadPool(100),n);
    run("virtual-per-task",Executors.newVirtualThreadPerTaskExecutor(),n);
  }
}
'''
work = pathlib.Path(tempfile.mkdtemp(prefix='loom-lab-'))
(work/'LoomCompare.java').write_text(source, encoding='utf-8')
compiled = subprocess.run(['javac', str(work/'LoomCompare.java')], capture_output=True, text=True)
print(compiled.stdout, compiled.stderr)
if compiled.returncode == 0:
    run = subprocess.run(['java','-cp',str(work),'LoomCompare'],capture_output=True,text=True,timeout=180)
    print(run.stdout, run.stderr)
    print('exit:',run.returncode)
    if run.returncode: raise RuntimeError('Java Loom comparison failed')
else:
    raise RuntimeError('Compile with JDK 21 or newer for virtual threads')

## Interpret carefully
The fixed-size platform pool is intentionally bounded; the virtual executor creates one virtual thread per task. The Java demo reports elapsed time, an approximate peak of live Java platform threads (ThreadMXBean does not count virtual threads as ordinary platform threads), heap usage before/after, and process CPU time when the JDK exposes it. These are diagnostic snapshots, not a benchmark or precise native RSS measurement. Run several times and record JDK version.

**Expected:** virtual threads can increase concurrency for blocking tasks with fewer platform threads; they should not magically speed CPU-bound work. Heap deltas are noisy and include unrelated objects.

**Why:** JEP 444 stores suspended virtual-thread stacks in heap chunks and schedules them over carrier platform threads. Heap retention, thread locals and downstream resource limits still matter. JDK 21+ required. Spark pools are not converted automatically.